In [0]:
%run ./00_common

In [0]:
import pandas as pd

def _overwrite_empty_table(table_name: str):
    empty_sdf = spark.table(table_name).limit(0)
    empty_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

def _write_pdf_to_delta(
    pdf: pd.DataFrame,
    table_name: str,
    select_exprs,
    mode: str = "overwrite",
    overwrite_schema: bool = True
):
    if pdf is None or len(pdf) == 0:
        if mode == "overwrite":
            return
        _overwrite_empty_table(table_name)
        return

    sdf = spark.createDataFrame(safe_pdf(pdf))
    sdf = sdf.selectExpr(*select_exprs)

    writer = sdf.write.format("delta").mode(mode)
    if overwrite_schema:
        writer = writer.option("overwriteSchema", "true")
    writer.saveAsTable(table_name)

def write_bronze_orders_delta(pdf: pd.DataFrame, mode: str = "overwrite"):
    _write_pdf_to_delta(
        pdf,
        BRONZE_ORDERS_TABLE,
        [
            "CAST(order_id AS BIGINT) AS order_id",
            "CAST(customer_id AS INT) AS customer_id",
            "CAST(order_date AS DATE) AS order_date",
            "CAST(amount AS DOUBLE) AS amount",
            "CAST(status AS STRING) AS status",
            "CAST(_ingestion_ts AS TIMESTAMP) AS _ingestion_ts",
            "CAST(_source_file AS STRING) AS _source_file",
            "CAST(_run_id AS STRING) AS _run_id",
            "CAST(_run_date AS STRING) AS _run_date"
        ],
        mode=mode
    )

def write_bronze_customers_delta(pdf: pd.DataFrame, mode: str = "overwrite"):
    _write_pdf_to_delta(
        pdf,
        BRONZE_CUSTOMERS_TABLE,
        [
            "CAST(customer_id AS INT) AS customer_id",
            "CAST(customer_name AS STRING) AS customer_name",
            "CAST(country AS STRING) AS country",
            "CAST(email AS STRING) AS email",
            "CAST(updated_at AS TIMESTAMP) AS updated_at",
            "CAST(_ingestion_ts AS TIMESTAMP) AS _ingestion_ts",
            "CAST(_source_file AS STRING) AS _source_file",
            "CAST(_run_id AS STRING) AS _run_id",
            "CAST(_run_date AS STRING) AS _run_date"
        ],
        mode=mode
    )

def write_reject_orders_delta(pdf: pd.DataFrame, mode: str = "overwrite"):
    _write_pdf_to_delta(
        pdf,
        REJECT_ORDERS_TABLE,
        [
            "CAST(order_id AS BIGINT) AS order_id",
            "CAST(customer_id AS INT) AS customer_id",
            "CAST(order_date AS DATE) AS order_date",
            "CAST(amount AS DOUBLE) AS amount",
            "CAST(status AS STRING) AS status",
            "CAST(_ingestion_ts AS TIMESTAMP) AS _ingestion_ts",
            "CAST(_source_file AS STRING) AS _source_file",
            "CAST(_run_id AS STRING) AS _run_id",
            "CAST(_run_date AS STRING) AS _run_date",
            "CAST(reject_reason AS STRING) AS reject_reason"
        ],
        mode=mode,
        overwrite_schema=True
    )

def write_silver_orders_delta(pdf: pd.DataFrame, mode: str = "overwrite"):
    _write_pdf_to_delta(
        pdf,
        SILVER_ORDERS_TABLE,
        [
            "CAST(order_id AS BIGINT) AS order_id",
            "CAST(customer_id AS INT) AS customer_id",
            "CAST(order_date AS DATE) AS order_date",
            "CAST(amount AS DOUBLE) AS amount",
            "CAST(status AS STRING) AS status",
            "CAST(_ingestion_ts AS TIMESTAMP) AS _ingestion_ts",
            "CAST(_source_file AS STRING) AS _source_file",
            "CAST(_run_id AS STRING) AS _run_id",
            "CAST(_run_date AS STRING) AS _run_date"
        ],
        mode=mode,
        overwrite_schema=True
    )

def write_silver_customers_delta(pdf: pd.DataFrame, mode: str = "overwrite"):
    _write_pdf_to_delta(
        pdf,
        SILVER_CUSTOMERS_TABLE,
        [
            "CAST(customer_id AS INT) AS customer_id",
            "CAST(customer_name AS STRING) AS customer_name",
            "CAST(country AS STRING) AS country",
            "CAST(email AS STRING) AS email",
            "CAST(updated_at AS TIMESTAMP) AS updated_at",
            "CAST(_ingestion_ts AS TIMESTAMP) AS _ingestion_ts",
            "CAST(_source_file AS STRING) AS _source_file",
            "CAST(_run_id AS STRING) AS _run_id",
            "CAST(_run_date AS STRING) AS _run_date"
        ],
        mode=mode,
        overwrite_schema=True
    )

def write_gold_daily_delta(pdf: pd.DataFrame, mode: str = "overwrite"):
    _write_pdf_to_delta(
        pdf,
        GOLD_DAILY_SALES_TABLE,
        [
            "CAST(order_date AS DATE) AS order_date",
            "CAST(total_orders AS BIGINT) AS total_orders",
            "CAST(total_revenue AS DOUBLE) AS total_revenue",
            "CAST(avg_ticket AS DOUBLE) AS avg_ticket"
        ],
        mode=mode,
        overwrite_schema=True
    )

def write_gold_country_delta(pdf: pd.DataFrame, mode: str = "overwrite"):
    _write_pdf_to_delta(
        pdf,
        GOLD_SALES_BY_COUNTRY_TABLE,
        [
            "CAST(order_date AS DATE) AS order_date",
            "CAST(country AS STRING) AS country",
            "CAST(total_orders AS BIGINT) AS total_orders",
            "CAST(total_revenue AS DOUBLE) AS total_revenue"
        ],
        mode=mode,
        overwrite_schema=True
    )

def append_events_delta(pdf: pd.DataFrame):
    _write_pdf_to_delta(
        pdf,
        OPS_PIPELINE_EVENTS,
        [
            "CAST(event_ts AS TIMESTAMP) AS event_ts",
            "CAST(level AS STRING) AS level",
            "CAST(run_id AS STRING) AS run_id",
            "CAST(stage AS STRING) AS stage",
            "CAST(dataset AS STRING) AS dataset",
            "CAST(status AS STRING) AS status",
            "CAST(message AS STRING) AS message",
            "CAST(rows_ok AS BIGINT) AS rows_ok",
            "CAST(rows_reject AS BIGINT) AS rows_reject",
            "CAST(error_code AS STRING) AS error_code",
            "CAST(extra AS STRING) AS extra"
        ],
        mode="append"
    )

def append_alerts_delta(pdf: pd.DataFrame):
    _write_pdf_to_delta(
        pdf,
        OPS_PIPELINE_ALERTS,
        [
            "CAST(alert_ts AS TIMESTAMP) AS alert_ts",
            "CAST(run_id AS STRING) AS run_id",
            "CAST(alert_type AS STRING) AS alert_type",
            "CAST(severity AS STRING) AS severity",
            "CAST(message AS STRING) AS message"
        ],
        mode="append"
    )

def append_dataset_versions_delta(pdf: pd.DataFrame):
    _write_pdf_to_delta(
        pdf,
        OPS_DATASET_VERSIONS,
        [
            "CAST(audit_ts AS TIMESTAMP) AS audit_ts",
            "CAST(run_id AS STRING) AS run_id",
            "CAST(table_name AS STRING) AS table_name",
            "CAST(version AS BIGINT) AS version",
            "CAST(operation AS STRING) AS operation",
            "CAST(operation_ts AS TIMESTAMP) AS operation_ts",
            "CAST(num_output_rows AS STRING) AS num_output_rows",
            "CAST(user_name AS STRING) AS user_name"
        ],
        mode="append"
    )